Eu tenho 100k newick na pasta mutation_dataset_100k_phylCNN

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ete3 import Tree
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split

c:\Users\JPC\anaconda3\envs\phylodeepenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\JPC\anaconda3\envs\phylodeepenv\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
TREE_DIR = r"mutation_dataset_100k_phyloCNN\\split_trees"
PARAM_FILE = r"mutation_dataset_100k_phyloCNN\\tabela_100k_phyloCNN.txt"

BAD_INDICES = {
    31035, 36174, 49680, 52494,
    53227, 73155, 81488, 97073
}

# no .nwk, a árvore dos ids de erro ta uma linha vazia, isso da problema
# acho melhor logo transformar o arquivo removendo as linhas vazias, e removendo os ids da tabela de parametros
# depois so precisa cuidar do v9, q vai 9992 arvores, mas qualquer coisa nbem treina dai

TARGET_COLUMNS = ["R_nought_1", "R_nought_2", "infectious_time"]

In [3]:
PARAM_FILE = r"mutation_dataset_100k_phyloCNN\tabela_100k_phyloCNN.txt"
MUTANTS_FILE = r"mutation_dataset_100k_phyloCNN\number_of_mutantes.npy"

BAD_INDICES = {
    31035, 36174, 49680, 52494, 53227,
    73155, 81488, 97073
}

# Load parameter table
params = pd.read_csv(PARAM_FILE, sep="\t")

# Load number of mutants for each original tree
number_of_mutantes = np.load(MUTANTS_FILE)

# Check alignment before removing bad trees
if len(params) != len(number_of_mutantes):
    raise ValueError(
        f"Parameter table has {len(params)} rows, "
        f"but number_of_mutantes has {len(number_of_mutantes)} entries."
    )

# Add number of mutants
params["number_of_mutantes"] = number_of_mutantes

# Remove malfunctioning trees
params = params[~params["index"].isin(BAD_INDICES)].copy()

# Reset dataframe index
params.reset_index(drop=True, inplace=True)

# Total number of nodes in a full binary tree:
# total_nodes = 2 * internal_nodes + 1
params["total_nodes"] = 2 * params["tree_size"] + 1

# Fraction of mutated nodes
params["mutation_fraction"] = (
    params["number_of_mutantes"] / params["total_nodes"]
)


# <= 5 mutated nodes -> R02 is not reliably estimable
few_mutations = params["number_of_mutantes"] <= 5

params.loc[few_mutations, "R_nought_2"] = (
    params.loc[few_mutations, "R_nought_1"]
)

# Case 2:
# >95% of nodes mutated -> R01 is not reliably estimable
mostly_mutated = params["mutation_fraction"] > 0.95

params.loc[mostly_mutated, "R_nought_1"] = (
    params.loc[mostly_mutated, "R_nought_2"]
)

print("Original number of parameter rows:", len(number_of_mutantes))
print("Removed bad trees:", len(BAD_INDICES))
print("Final parameter rows:", len(params))

print("Trees with <= 5 mutations:", few_mutations.sum())
print("Trees with >95% mutated nodes:", mostly_mutated.sum())

print("\nBad indices still present:",
      sorted(set(params["index"]) & BAD_INDICES))


Original number of parameter rows: 100000
Removed bad trees: 8
Final parameter rows: 99992
Trees with <= 5 mutations: 25985
Trees with >95% mutated nodes: 4962

Bad indices still present: []


In [4]:
params_by_id = params.set_index("index")

In [5]:
def ete_to_pyg(tree, target):
    nodes = list(tree.traverse())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    node_features = []
    edges = []

    for node in nodes:
        node_features.append([
            float(node.DIST_TO_START),
            float(node.is_leaf())
        ])

        if node.up is not None:
            parent = node_to_idx[node.up]
            child = node_to_idx[node]

            edges.append([parent, child])
            edges.append([child, parent])

    x = torch.tensor(node_features, dtype=torch.float32)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()



    #make target [1, 3], not [3]
    y = torch.tensor(target, dtype=torch.float32).unsqueeze(0)

    return Data(x=x, edge_index=edge_index, y=y)

In [6]:
def process_nwk(part_number, params_by_id):
    filename = os.path.join(TREE_DIR, f"trees_part_{part_number}.nwk")
    start_index = part_number * 10000
    dataset = []

    with open(filename, "r") as f:
        for local_index, line in enumerate(f):
            global_index = start_index + local_index

            if global_index in BAD_INDICES:
                continue

            line = line.strip()

            if not line:
                continue

            tree = Tree(line, format=1)

            target = params_by_id.loc[global_index, TARGET_COLUMNS].to_numpy(dtype=np.float32)

            graph = ete_to_pyg(tree, target)
            graph.tree_id = global_index

            dataset.append(graph)

    print(f"Part {part_number}: {len(dataset)} valid trees")

    return dataset

In [7]:
dataset = process_nwk(2, params_by_id)

Part 2: 9999 valid trees


In [8]:
print(dataset[0])
print("Node features:", dataset[0].x.shape)
print(dataset[0].x[:5])

Data(x=[997, 2], edge_index=[2, 1992], y=[1, 3], tree_id=20000)
Node features: torch.Size([997, 2])
tensor([[0.0585, 0.0000],
        [0.5998, 0.0000],
        [2.0586, 0.0000],
        [2.2627, 0.0000],
        [0.7980, 0.0000]])


In [9]:
def split_dataset(dataset):
    indices = np.arange(len(dataset))

    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    train_dataset = [dataset[i] for i in train_idx]
    val_dataset = [dataset[i] for i in val_idx]
    test_dataset = [dataset[i] for i in test_idx]

    return train_dataset, val_dataset, test_dataset

In [10]:
train_dataset, val_dataset, test_dataset = split_dataset(dataset)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

7999
1000
1000


In [11]:
def normalize_targets(dataset, target_mean, target_std):

    for graph in dataset:
        graph.y = (graph.y - target_mean)/target_std

    return dataset

In [12]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Descomentar no treino seguitne
checkpoint = torch.load("modelos\\GNN1\\v1.pt", map_location=device, weights_only=False)

target_mean = torch.tensor(checkpoint["target_mean"], dtype=torch.float32)

target_std = torch.tensor(checkpoint["target_std"], dtype=torch.float32)



In [13]:
train_y = torch.cat([g.y for g in train_dataset], dim=0)

target_mean = train_y.mean(dim=0)
target_std = train_y.std(dim=0).clamp_min(1e-8)

print("Target mean:", target_mean)
print("Target std:", target_std)

Target mean: tensor([3.1761, 5.9279, 5.4916])
Target std: tensor([1.4304, 3.2058, 2.5830])


In [14]:
train_dataset = normalize_targets(train_dataset, target_mean, target_std)
val_dataset = normalize_targets(val_dataset, target_mean, target_std)
test_dataset = normalize_targets(test_dataset, target_mean, target_std)

In [15]:
class PhyloGNN(nn.Module):
    def __init__(self, num_features=2):
        super().__init__()

        self.gnn1 = GATConv(num_features, 64, heads=4, concat=False)
        self.gnn2 = GATConv(64, 64, heads=4, concat=False)
        self.gnn3 = GATConv(64, 64, heads=4, concat=False)

        self.regressor = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

    def forward(self, x, edge_index, batch):
        x = torch.relu(self.gnn1(x, edge_index))
        x = torch.relu(self.gnn2(x, edge_index))
        x = torch.relu(self.gnn3(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)

        x = torch.cat([x_mean, x_max], dim=1)

        return self.regressor(x)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
model = PhyloGNN(num_features=2).to(device)

#descomentar no treino seguinte
model.load_state_dict(checkpoint["model_state_dict"])

model.train()

PhyloGNN(
  (gnn1): GATConv(2, 64, heads=4)
  (gnn2): GATConv(64, 64, heads=4)
  (gnn3): GATConv(64, 64, heads=4)
  (regressor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=3, bias=True)
  )
)

In [18]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        pred = model(batch.x, batch.edge_index, batch.batch)

        loss = criterion(pred, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [19]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [21]:
batch = next(iter(train_loader))

batch = batch.to(device)

pred = model(batch.x, batch.edge_index, batch.batch)

print("pred:", pred.shape)
print("target:", batch.y.shape)

pred: torch.Size([32, 3])
target: torch.Size([32, 3])


In [22]:
best_val_loss = float("inf")

for epoch in range(1, 25):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch:03d} | Train: {train_loss:.5f} | Val: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

        torch.save({
                "model_state_dict": model.state_dict(),
                # These are the V0 normalization statistics.
                "target_mean": target_mean.cpu().numpy(),
                "target_std": target_std.cpu().numpy(),

                "best_val_loss": best_val_loss,
                "epoch": best_epoch}, "modelos\\GNN1\\v2.pt")

Epoch 001 | Train: 0.26199 | Val: 0.24741
Epoch 002 | Train: 0.25004 | Val: 0.25267
Epoch 003 | Train: 0.24567 | Val: 0.23294
Epoch 004 | Train: 0.24113 | Val: 0.22942


KeyboardInterrupt: 

In [23]:
best_checkpoint = torch.load("modelos\\GNN1\\v2.pt", map_location=device, weights_only=False)

model.load_state_dict(best_checkpoint["model_state_dict"])

model.eval()

print("Best epoch:",best_checkpoint["epoch"])

print("Best validation loss:", best_checkpoint["best_val_loss"])

Best epoch: 4
Best validation loss: 0.22942280209064483


In [25]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()

    predictions = []
    true_values = []

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch.x, batch.edge_index, batch.batch)

        predictions.append(pred.cpu().numpy())
        true_values.append(batch.y.cpu().numpy())

    return np.concatenate(predictions), np.concatenate(true_values)

In [26]:
pred_scaled, y_scaled = predict(model, test_loader, device)

In [27]:
mean_np = target_mean.cpu().numpy()
std_np = target_std.cpu().numpy()

pred = pred_scaled * std_np + mean_np
y_true = y_scaled * std_np + mean_np

In [28]:
from sklearn.metrics import mean_absolute_error

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    mae = mean_absolute_error(y_true[:, i], pred[:, i])
    print(f"{name}: MAE = {mae:.4f}")

R01: MAE = 0.4784
R02: MAE = 1.3677
Infectious time: MAE = 0.4391


In [29]:
from sklearn.metrics import mean_squared_error

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    rmse = mean_squared_error(y_true[:, i], pred[:, i]) ** 0.5
    print(f"{name}: RMSE = {rmse:.4f}")

R01: RMSE = 0.8470
R02: RMSE = 1.7581
Infectious time: RMSE = 0.5852


In [30]:
import numpy as np

mape = np.mean(
    np.abs((y_true - pred) / y_true),
    axis=0
) * 100

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    print(f"{name}: MAPE = {mape[i]:.2f}%")

R01: MAPE = 15.18%
R02: MAPE = 28.46%
Infectious time: MAPE = 8.93%
